In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout


Data Collection 

train_ds = tf.keras.utils.image_dataset_from_directory(
    r"/kaggle/input/solar-panel-preprocessed/Faulty_solar_panel",
    labels = 'inferred',
    label_mode = "categorical",
    image_size = (256, 256),
    batch_size = 32,
    validation_split = 0.2,
    subset = 'training',
    seed = 123
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    r"/kaggle/input/solar-panel-preprocessed/Faulty_solar_panel",
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(256, 256),
    batch_size=32,
    label_mode='categorical'
)

Loading Base Model

In [ ]:
image_height, image_width = 256, 256
num_class = 6
batch_size = 32

base_model = keras.applications.ResNet50(
    weights = 'imagenet',
    include_top = False,
    pooling = 'max',
    classes = num_class,
    input_shape = (image_height, image_width, 3)
)

for layer in base_model.layers[-30:]:
    layer.trainable = True

Custom Layer in pretrained model

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2)
])

resnet_model = Sequential()
resnet_model.add(data_augmentation)
resnet_model.add(base_model)
resnet_model.add(Flatten())
resnet_model.add(Dense(1024, activation = 'relu'))
resnet_model.add(Dropout(0.2))
resnet_model.add(Dense(num_class, activation = "softmax"))

Model Summary 

In [ ]:
resnet_model.summary()

Compiling Model 

In [ ]:
resnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


Training Model 

In [ ]:
history = resnet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
)


Saving Model 

In [ ]:
resnet_model.save("solar_panel_analyzer.keras") 